# Calibration Example

Load `calib.h5`, process the 1 kHz octave band, and print the detected calibration value.

In [ ]:
# imports 
from tapy.devices.sinus import Apollo
from tapy.bindings.acoular import SinusSamplesGenerator

In [ ]:
# instantiate the samples generator for the Apollo device

# the inventory number can be found on the device label, e.g. "TA181"
apollo = Apollo(inventory_no="TA181")

# enable analog channels 1 and 2
for ch in apollo.analog_inputs:
    ch.ENABLED = '1'
    ch.set_settings()
    print(ch.SampleRate) 

    sample_freq = float(ch.SampleRate)


In [ ]:
# Turn on ICP

apollo.analog_inputs[0].ICP = '2mA'
apollo.analog_inputs[0].sensitivity = 1.0 # record signals in V
apollo.analog_inputs[0].set_settings()

In [ ]:
source = SinusSamplesGenerator(device=apollo, _sample_freq=sample_freq, num_samples=-1, _num_channels=4)#
source.__dict__

res = source.result(num=256)
print(next(res).shape)


In [ ]:
import acoular as ac
import spectacoular as sp
import numpy as np

# CalibHelper class can be replaced by 
calib = ac.Calib(source=source, data=np.ones(1,4)) # dummy calibration factors (one could use a generic sensitivity here)
filt = sp.FiltOctave(source=calib, band=1000.0) # filter the signal to the 1kHz octave band
pow = ac.TimePower(source=filt) # compute power of the signal
avg = ac.Average(source=pow, num_per_average=512) # average over 512 samples
wh5 = ac.WriteH5(source=avg)
calib = sp.CalibHelper(source=wh5, buffer_size=100, delta=50) #


### Retrieve calibration values blockwise (512 samples per block) and print the detected calibration value for the block.

In [ ]:
if True:
    for data in calib.result(1):
        print(data.shape) # calib helper simply passes the data through
        print(calib.calibdata[:1, 0])  # print the detected calibration value and caliblevel
